In [6]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [1]:
%cd drive/MyDrive/project/semantic-segmentation-roads/eomt

/content/drive/.shortcut-targets-by-id/1kGi4cSNJjM14ClVvPXJ9VY70JDkofHGn/project/semantic-segmentation-roads/eomt


In [ ]:
!pip install -r requirements.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.4/219.4 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 67.0 MB

In [2]:
import os
import sys
import torch

PROJECT_DIR = "/content/drive/MyDrive/project/semantic-segmentation-roads"
EOMT_DIR = f"{PROJECT_DIR}/eomt"

state_dict_path_cs = "/content/drive/MyDrive/project/models_weights/eomt_cityscapes.bin"
state_dict_path_coco = "/content/drive/MyDrive/project/models_weights/eomt_coco.bin"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

os.chdir(PROJECT_DIR)

sys.path.insert(0, EOMT_DIR)     # for models.*, datasets.*, training.*
sys.path.insert(0, PROJECT_DIR)  # for eomt.*, utils.*

In [3]:
from eomt.semantic_eval import evaluate_semantic
from utils.model_loading import get_config, build_model, load_weights
from utils.data_loading import build_datamodule

## Modello A (Cityscapes)

In [4]:
config_cs = get_config()
data = build_datamodule(config_cs)
eval_dataset_cs, img_size_cs, num_classes_cs = data.val_dataloader(), data.img_size, data.num_classes
model_cs = build_model(config_cs, img_size_cs, num_classes_cs).eval().to(device)
model_cs = load_weights(model_cs, state_dict_path_cs, device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


Missing keys: []
Unexpected keys: []


In [5]:
evaluate_semantic(model_cs, eval_dataset_cs, device)

Evaluating on 500 images...


Eval:   0%|          | 0/500 [00:00<?, ?it/s]


mIoU: 81.68%


0.8167969584465027

## Modello (COCO)

In [6]:
config_coco = get_config(True)
data = build_datamodule(config_coco)
eval_dataset_coco, img_size_coco, num_classes_coco = data.val_dataloader(), data.img_size, data.num_classes
model_coco = build_model(config_coco, img_size_coco, num_classes_coco, True).eval().to(device)
model_coco = load_weights(model_coco, state_dict_path_coco, device)

Missing keys: []
Unexpected keys: []


In [7]:
evaluate_semantic(model_coco, eval_dataset_coco, device, True)

Evaluating on 500 images...


Eval:   0%|          | 0/500 [00:00<?, ?it/s]


mIoU: 51.79%


0.5179237127304077